# w03_data_contract.ipynb

## User Intent Lane — Data Contract

This notebook defines my lane's slice of the FlyRank warehouse and proves I understand the data.

## 1. The Contract — My Lane's Slice

**My Lane:** User Intent / Refresh Opportunity Scoring

### 1. What one row means for my lane:
One row = one content item (`content_hash_id`) for one day (`report_date`).

### 2. Which table(s) I'll use:
`fact_daily_sample` — the sample table (1 month of data) for development.

### 3. Which time window:
- **Development/validation:** `report_date` in `2026-03` (mid-panel)
- **Sealed test:** `report_date` in `2026-06` (final month — the `_sample` table)

### 4. What I'd predict (label/proxy):
`high_intent_label` = 1 if CTR > 0.05 AND `ga4_engaged_sessions` > 0 in the following month.

### 5. One thing I deliberately exclude:
I exclude pages with less than 10 impressions (`gsc_impressions < 10`) — they don't have enough signal to predict reliably.

## 2. Three Verification Queries

These queries prove I understand the data grain, row count, and availability.

In [ ]:
# Connect to the warehouse using getpass (no secrets needed)
import duckdb
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
SAMPLE = f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')"

print("✅ Connected to Hugging Face!")

In [ ]:
# QUERY 1: Prove the grain (one row = one content item + one day)
query1 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content,
        COUNT(DISTINCT report_date) AS unique_days
    FROM {SAMPLE}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

print("QUERY 1: Grain check (one row = one content item + one day)")
display(query1)

In [ ]:
# QUERY 2: Row count and date span for my slice (corrected columns)
query2 = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content
    FROM {SAMPLE}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
      AND gsc_impressions >= 10
""").df()

print("QUERY 2: Row count and date span (impressions >= 10)")
display(query2)

In [ ]:
# QUERY 3: Availability — filter with IS TRUE (corrected columns)
query3 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS has_gsc,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_ga4,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_both
    FROM {SAMPLE}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

print("QUERY 3: Availability (gsc_data_available, ga4_data_available)")
display(query3)

## 3. Five Features

Features I'll use to predict user intent:

| Feature | "Available when?" |
|---------|-------------------|
| `avg_position` | Knowable at decision moment — based on historical position data from GSC |
| `ctr` | Knowable at decision moment — based on historical clicks/impressions |
| `engagement_rate` | Knowable at decision moment — based on historical GA4 engaged sessions |
| `content_age_days` | Knowable at decision moment — based on content creation date |
| `impressions_90d` | Knowable at decision moment — based on historical impression data |

In [ ]:
# Build a small feature frame for my lane from the sample (corrected columns)
features_df = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        AVG(ga4_engaged_sessions) AS engagement_rate,
        -- content_age_days is in dim_content; we'll join it later
        SUM(gsc_impressions) AS impressions_90d
    FROM {SAMPLE}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 10
    LIMIT 10
""").df()

print("Five Features — Sample Frame")
display(features_df)

## 4. The Trap — Deliberate Leakage Experiment

**Warning: This demonstrates leakage! Do NOT keep this column!**

In [ ]:
# DELIBERATE LEAK — DO NOT KEEP
# This uses future outcome (CTR from the same month) as a feature

leak_df = con.sql(f"""
    SELECT
        content_hash_id,
        gsc_impressions,
        -- LEAK: using current CTR to predict future CTR (THIS IS CHEATING!)
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS future_ctr_leak
    FROM {SAMPLE}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 10
    LIMIT 10
""").df()

print("⚠️ LEAK DETECTED: 'future_ctr_leak' uses future data!")
print("This would give unrealistically perfect predictions. DELETE this column.")
display(leak_df)

# Clean up — drop the leaked column
leak_df = leak_df.drop(columns=['future_ctr_leak'])
print("\n✅ Cleaned: Removed leaked column. This is the honest version.")
display(leak_df)

## 4. One Named Limitation of My Slice

**Limitation:** My slice only includes pages with at least 10 impressions in the sample month. This is a deliberate choice to filter out noise, but it introduces bias toward more popular content. Low-impression pages (the long tail) are excluded, which means the model may not generalize to newly published or niche content.

This is a directional, not causal, analysis.

## 5. Self-Check

✅ I've named my lane: User Intent / Refresh Opportunity Scoring

✅ I've defined one row: content_hash_id + report_date

✅ I've picked a time window: 2026-03 (mid-panel)

✅ I've named my label/proxy: high_intent_label (CTR > 0.05 AND ga4_engaged_sessions > 0)

✅ I've excluded: pages with < 10 impressions

✅ I've run 3 verification queries

✅ I've built 5 features with "available when?" lines

✅ I've shown the leakage trap and deleted it

✅ I've named one limitation of my slice